# Cropchain Phase 1  
## Experiment 2: System Design and Implementation

### Overview
This experiment extends the Cropchain project using a structured Waterfall development approach. Building on the outputs from Experiment 1, this phase focuses on translating requirements into system-level design and implementation strategies.

### Objective
The goal of this experiment is to:
- Transform requirements into a system architecture
- Define technical components and workflows
- Generate implementation-level outputs using AI agents

### Methodology
We use a multi-agent architecture powered by `gpt-4o-mini`, where each agent represents a role in the software development lifecycle:
- System Engineer → defines architecture and system components
- Software Engineer → generates implementation strategy and technical breakdown

All outputs are generated sequentially to ensure consistency with Experiment 1 artifacts.

### Input Dependencies

This experiment uses outputs generated from Experiment 1:

- Project Manager Summary  
- Requirements Specification  

These files are loaded from:
src/outputs/experiment_1/


This ensures continuity in the Waterfall pipeline:

Requirements → Design → Implementation


In [17]:
import os
from pathlib import Path
from dotenv import load_dotenv
import autogen

CURRENT_DIR = Path.cwd()
BASE_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

load_dotenv(BASE_DIR / ".env", override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing. Add it to your .env file before running.")

MODEL_NAME = "gpt-4o-mini"

TEAM_LLM_CONFIG = {
    "config_list": [
        {
            "model": MODEL_NAME,
            "api_key": OPENAI_API_KEY,
        }
    ],
    "temperature": 0.2,
    "timeout": 120,
}

PROJECT_NAME = "Cropchain: Intelligent Farm-to-Table Supply Chain Management System"

EXP1_DIR = BASE_DIR / "src" / "outputs" / "experiment_1"
OUTPUT_DIR = BASE_DIR / "src" / "outputs" / "experiment_2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PM_SUMMARY_FILE = EXP1_DIR / "03_project_manager_summary.md"
REQ_OUTPUT_FILE = EXP1_DIR / "04_requirements_engineer_output.md"

for f in [PM_SUMMARY_FILE, REQ_OUTPUT_FILE]:
    if not f.exists():
        raise FileNotFoundError(f"Required Experiment 1 file is missing: {f}")

pm_summary_text = PM_SUMMARY_FILE.read_text(encoding="utf-8")
requirements_text = REQ_OUTPUT_FILE.read_text(encoding="utf-8")

print("Experiment 2 environment ready.")
print("Model:", MODEL_NAME)
print("Loaded PM summary:", bool(pm_summary_text.strip()))
print("Loaded requirements output:", bool(requirements_text.strip()))

Experiment 2 environment ready.
Model: gpt-4o-mini
Loaded PM summary: True
Loaded requirements output: True


## System Design Phase

In this step, the System Engineer agent translates the requirements into a high-level architecture.

This includes:
- System components
- Data flow
- Integration between stakeholders (farmers, restaurants, retailers)
- Scalability and real-time tracking considerations

The output defines how the Cropchain system will operate end-to-end.

In [18]:
project_manager_agent = autogen.ConversableAgent(
    name="Project_Manager",
    system_message="""
You are the Project Manager for a Waterfall software planning process.
Make every output structured, academic, and presentation-ready.
Always ask for deliverables, review tasks, rework tasks, productivity, effort, assumptions, risks, and open questions.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

system_engineer_agent = autogen.ConversableAgent(
    name="System_Engineer",
    system_message="""
You are the System Engineer for Cropchain.
Create a detailed design output that includes:
- architecture overview
- actor interactions
- data model ideas
- modules and interfaces
- security and traceability design
- logistics and analytics considerations
- design tasks
- review tasks
- rework tasks
- productivity and effort estimates
Design architecture to strictly meet non-functional requirements (GDPR, <2s latency, 99.9% uptime, 10k concurrent users).
Align your effort estimates with the Project Manager's total hours.
Use 5 pages completed per day unless instructed otherwise.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

software_engineer_agent = autogen.ConversableAgent(
    name="Software_Engineer",
    system_message="""
You are the Software Engineer for Cropchain.
Create an implementation planning output that includes:
- implementation modules
- backend and frontend breakdown
- API and database components
- AI service components for prediction and pricing
- traceability and alerting features
- review tasks
- rework tasks
- Feature Point estimates
- productivity and effort estimates
Use PostgreSQL as the primary database so the implementation stays consistent with the System Engineer architecture.
Address how the backend and API will specifically handle the strict non-functional constraints (GDPR, scale, uptime).
Align your effort estimates directly with the Project Manager's timeline.
Use 5 Feature Points completed per day instead of SLOC.
Do not use SLOC as a metric.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

print("Experiment 2 agents created successfully.")

[autogen.oai.client: 03-26 21:59:33] {164} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-26 21:59:33] {164} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 03-26 21:59:33] {164} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
Experiment 2 agents created successfully.


### Design Interpretation

The generated system architecture aligns well with the Cropchain platform by incorporating:

- Multiple stakeholders (farmers, restaurants, retailers)
- Real-time supply tracking and logistics coordination
- Predictive analytics for crop yield and pricing
- Integration between ordering, inventory, and shipment systems

This design reflects a scalable and modular system suitable for a real-world farm-to-table supply chain platform.

## Implementation Planning Phase

The Software Engineer agent builds on the system design and defines the implementation strategy.

This includes:
- Technology stack recommendations
- Backend and frontend structure
- API design and integration
- Data storage and processing approach

This step bridges system design with practical implementation.

In [19]:
project_manager_to_system_engineer_prompt = f"""
The Project Manager summary and Requirements Engineer output for "{PROJECT_NAME}" are provided below.

=== PROJECT MANAGER SUMMARY ===
{pm_summary_text}

=== REQUIREMENTS ENGINEER OUTPUT ===
{requirements_text}

Using these baseline Waterfall artifacts, create a detailed system design output for Cropchain.

Your output must include:
1. design phase overview
2. design deliverables
3. architecture components
4. major actors and interactions
5. key interfaces and data entities
6. support for yield prediction, fair pricing, logistics visibility, alerts, and traceability
7. design tasks and activities
8. review tasks
9. rework tasks
10. productivity estimate using 5 pages per day
11. effort estimate in days
12. specific architecture for non-functional requirements (10k users, <2s latency, 99.9% uptime, GDPR)
13. assumptions, risks, and open questions

Make the response structured and presentation-ready.
""".strip()

project_manager_to_software_engineer_prompt = f"""
The System Engineer must continue from the Experiment 1 Waterfall baseline for "{PROJECT_NAME}".

=== PROJECT MANAGER SUMMARY ===
{pm_summary_text}

=== REQUIREMENTS ENGINEER OUTPUT ===
{requirements_text}

Keep the implementation consistent with the System Engineer output, especially by using PostgreSQL as the primary relational database.

Create the implementation planning output for Cropchain.

Your output must include:
1. implementation phase overview
2. implementation modules
3. frontend and backend breakdown
4. database and API components
5. AI and analytics service components
6. tasks and activities
7. review tasks
8. rework tasks
9. total Feature Point estimate
10. productivity estimate using 5 Feature Points per day
11. effort estimate in days
12. specific architecture for non-functional requirements (10k users, <2s latency, 99.9% uptime, GDPR)
13. assumptions, risks, and open questions

Make the response structured and presentation-ready.
""".strip()

print("Experiment 2 prompts prepared.")

Experiment 2 prompts prepared.


### Implementation Interpretation

The implementation plan translates the system architecture into a practical development strategy by defining:

- Backend services for data processing and supply chain logic
- APIs for communication between stakeholders
- Frontend interfaces for farmers, restaurants, and retailers
- Data storage for inventory, pricing, and tracking

This step bridges design and actual development, making the system feasible to build.

## Experiment 2 Implementation

This section generates the actual System Engineer and Software Engineer outputs using the Experiment 1 baseline artifacts.

In [20]:
def save_text(filename: str, text: str):
    file_path = OUTPUT_DIR / filename
    file_path.write_text(text, encoding="utf-8")
    print(f"Saved: {file_path}")

def save_chat_history(filename: str, chat_result):
    file_path = OUTPUT_DIR / filename
    if hasattr(chat_result, "chat_history"):
        lines = []
        for item in chat_result.chat_history:
            role = item.get("name", item.get("role", "Unknown"))
            content = item.get("content", "")
            lines.append(f"{role}:\n{content}\n")
        file_path.write_text(("\n" + ("-" * 80) + "\n").join(lines), encoding="utf-8")
        print(f"Saved chat history: {file_path}")
    else:
        print("No chat_history found on chat result.")

In [21]:
def run_experiment_2():
    print("=" * 80)
    print("STEP 1: Project Manager -> System Engineer")
    print("=" * 80)

    pm_to_system = project_manager_agent.initiate_chat(
        recipient=system_engineer_agent,
        message=project_manager_to_system_engineer_prompt,
        max_turns=1,
    )

    print("\n" + "=" * 80)
    print("STEP 2: Project Manager -> Software Engineer")
    print("=" * 80)

    pm_to_software = project_manager_agent.initiate_chat(
        recipient=software_engineer_agent,
        message=project_manager_to_software_engineer_prompt,
        max_turns=1,
    )

    return pm_to_system, pm_to_software

In [22]:
pm_to_system_result, pm_to_software_result = run_experiment_2()

STEP 1: Project Manager -> System Engineer
Project_Manager (to System_Engineer):

The Project Manager summary and Requirements Engineer output for "Cropchain: Intelligent Farm-to-Table Supply Chain Management System" are provided below.

=== PROJECT MANAGER SUMMARY ===
# Project Plan for Cropchain Web-Based Platform

## 1. Requirements Phase

### Deliverables:
- Requirements Specification Document
- Use Case Diagrams
- User Stories

### Tasks and Activities:
- Conduct stakeholder interviews to gather requirements.
- Define user personas for farmers, restaurants, and grocery stores.
- Document functional and non-functional requirements.
- Create use case diagrams to visualize interactions.

### Review Tasks:
- Review requirements with stakeholders for validation.
- Conduct a requirements walkthrough with the project team.

### Rework Tasks:
- Revise requirements based on stakeholder feedback.
- Update use case diagrams as necessary.

### Productivity Assumptions:
- 2 weeks for gathering

In [23]:
save_chat_history("01_pm_to_system_engineer_chat.txt", pm_to_system_result)
save_chat_history("02_pm_to_software_engineer_chat.txt", pm_to_software_result)

if hasattr(pm_to_system_result, "chat_history") and pm_to_system_result.chat_history:
    system_output = pm_to_system_result.chat_history[-1].get("content", "")
    save_text("03_system_engineer_output.md", system_output)

if hasattr(pm_to_software_result, "chat_history") and pm_to_software_result.chat_history:
    software_output = pm_to_software_result.chat_history[-1].get("content", "")
    save_text("04_software_engineer_output.md", software_output)

print("Experiment 2 artifacts saved.")

Saved chat history: /Users/shai/Desktop/School/ProjectManagement/Cropchain/src/outputs/experiment_2/01_pm_to_system_engineer_chat.txt
Saved chat history: /Users/shai/Desktop/School/ProjectManagement/Cropchain/src/outputs/experiment_2/02_pm_to_software_engineer_chat.txt
Saved: /Users/shai/Desktop/School/ProjectManagement/Cropchain/src/outputs/experiment_2/03_system_engineer_output.md
Saved: /Users/shai/Desktop/School/ProjectManagement/Cropchain/src/outputs/experiment_2/04_software_engineer_output.md
Experiment 2 artifacts saved.


In [24]:
required_files = [
    OUTPUT_DIR / "03_system_engineer_output.md",
    OUTPUT_DIR / "04_software_engineer_output.md",
]

for f in required_files:
    print(f"{f.name}: {'FOUND' if f.exists() else 'MISSING'}")

03_system_engineer_output.md: FOUND
04_software_engineer_output.md: FOUND


## Final Analysis

Experiment 2 successfully extends the Cropchain project from requirements into system design and implementation planning.

### Key Achievements
- Converted requirements into a structured system architecture
- Defined clear component interactions and workflows
- Generated implementation-ready technical strategies
- Maintained consistency with Experiment 1 outputs

### Strengths
- Strong alignment with real-world supply chain systems
- Clear separation of design and implementation responsibilities
- Reusable outputs for further development phases

### Limitations
- Generated architecture is conceptual and may require refinement for production
- Technology choices are generalized and may vary in real deployment

### Conclusion
This experiment establishes a solid technical foundation for the Cropchain platform and prepares the project for evaluation and refinement in the next phase.